# 06 — Final Comparison and Report
Consolidates every model from Phase 1, Phase 2, and Phase 3 onto a single comparison table evaluated on the same NSL-KDD test set, renders the publication-style architecture diagrams, plots the final cross-phase ROC, and emits `results/report.md`.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import seaborn as sns

sys.path.insert(0, os.path.abspath('.'))
from utils.config import (
    RESULTS_DIR, FIGURES_DIR, PHASE1_RESULTS, PHASE2_RESULTS,
)
from utils.data_loader import load_splits
from utils.evaluation import plot_roc_curves, plot_metric_bars

sns.set_style('whitegrid')

## 1. Cross-phase comparison table

In [ ]:
frames = []

p1_path = os.path.join(PHASE1_RESULTS, 'baseline_results.csv')
if os.path.exists(p1_path):
    p1 = pd.read_csv(p1_path)
    p1['phase'] = 'Phase 1'
    frames.append(p1)

p2_path = os.path.join(PHASE2_RESULTS, 'full_comparison.csv')
if os.path.exists(p2_path):
    p2 = pd.read_csv(p2_path)
    p2['phase'] = p2['model'].apply(
        lambda m: 'Phase 1' if 'Phase 1' in str(m) else 'Phase 2')
    # Avoid double-counting Phase 1 rows present in both CSVs.
    if 'p1' in locals():
        p2 = p2[p2['phase'] != 'Phase 1']
    frames.append(p2)

p3 = pd.read_csv(os.path.join(RESULTS_DIR, 'ablation_phase3.csv'))
p3['phase'] = 'Phase 3'
frames.append(p3)

full = pd.concat(frames, ignore_index=True)
full = full[['phase', 'model', 'accuracy', 'precision', 'recall', 'f1', 'auc']]
full = full.sort_values(['phase', 'f1'], ascending=[True, False]).reset_index(drop=True)
full.to_csv(os.path.join(RESULTS_DIR, 'phase3_full_comparison.csv'), index=False)
full.style.format({c: '{:.4f}' for c in ['accuracy','precision','recall','f1','auc']})

## 2. Final ROC across phases

In [ ]:
splits = load_splits(os.path.join(RESULTS_DIR, 'processed_data.npz'))
y_te = splits['y_test']

modelA = np.load(os.path.join(RESULTS_DIR, 'modelA_scores.npz'))
modelC = np.load(os.path.join(RESULTS_DIR, 'modelC_scores.npz'))

plot_roc_curves(
    [
        ('Autoencoder only',       y_te, modelA['ae_test_scores'],  '#3498db'),
        ('Isolation Forest (raw)', y_te, modelA['if_test_scores'],  '#f39c12'),
        ('Deep IF (Model A)',      y_te, modelA['dif_test_scores'], '#9b59b6'),
        ('Cascaded AE+XGB (Model C)', y_te, modelC['cax_binary_scores'], '#16a085'),
    ],
    title='Final cross-phase ROC on NSL-KDD test set',
    path=os.path.join(FIGURES_DIR, 'phase3_roc_final.png'),
)

## 3. Architecture diagrams (publication-style)
Drawn with matplotlib so they regenerate every time the notebook runs.

In [ ]:
def _box(ax, x, y, w, h, text, fc, ec='black', fs=10, fw='normal'):
    p = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.04',
                       linewidth=1.5, edgecolor=ec, facecolor=fc)
    ax.add_patch(p)
    ax.text(x + w / 2, y + h / 2, text, ha='center', va='center',
            fontsize=fs, fontweight=fw)

def _arrow(ax, x1, y1, x2, y2, label=None):
    a = FancyArrowPatch((x1, y1), (x2, y2),
                        arrowstyle='-|>', mutation_scale=15,
                        linewidth=1.5, color='#444')
    ax.add_patch(a)
    if label:
        ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.12, label,
                ha='center', fontsize=9, style='italic', color='#444')

ML_FC = '#fff3cd'   # light yellow — ML
DL_FC = '#d6eaff'   # light blue   — DL
OUT_FC = '#d4edda'  # light green  — output

In [ ]:
# ---- Model A — Deep Isolation Forest --------------------------------------
fig, ax = plt.subplots(figsize=(13, 4))
ax.set_xlim(0, 13); ax.set_ylim(0, 4); ax.axis('off')

_box(ax, 0.2, 1.5, 1.6, 1.2, 'Input record\n(43 dims)\n[B, 43]', '#f8f9fa')
_box(ax, 2.4, 0.4, 3.2, 3.2,
     'Autoencoder Encoder (DL)\nLinear→128→64→32\nBN + LeakyReLU + Dropout',
     DL_FC, fw='bold')
_box(ax, 6.3, 1.5, 1.7, 1.2, 'latent z\n[B, 32]', '#f8f9fa')
_box(ax, 8.6, 0.4, 2.6, 3.2,
     'Isolation Forest (ML)\n200 iTrees on z\nscore = −E[path-len]',
     ML_FC, fw='bold')
_box(ax, 11.6, 1.5, 1.2, 1.2, 'anomaly\nscore', OUT_FC, fw='bold')

_arrow(ax, 1.8, 2.1, 2.4, 2.1)
_arrow(ax, 5.6, 2.1, 6.3, 2.1, 'forward')
_arrow(ax, 8.0, 2.1, 8.6, 2.1)
_arrow(ax, 11.2, 2.1, 11.6, 2.1)

ax.text(6.5, 0.05,
        'DL extracts; ML decides — IF replaces the AE\'s brittle MSE threshold with a path-length statistic',
        ha='center', fontsize=10, style='italic', color='#555')
ax.set_title('Model A — Deep Isolation Forest', fontsize=13, fontweight='bold')

# Legend
leg = [mpatches.Patch(color=DL_FC, label='Deep Learning'),
       mpatches.Patch(color=ML_FC, label='Machine Learning'),
       mpatches.Patch(color=OUT_FC, label='Output')]
ax.legend(handles=leg, loc='upper right', fontsize=9, frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'diagram_model_A.png'),
            dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
# ---- Model C — Cascaded AE + XGBoost --------------------------------------
fig, ax = plt.subplots(figsize=(13, 6))
ax.set_xlim(0, 13); ax.set_ylim(0, 6); ax.axis('off')

_box(ax, 0.2, 2.5, 1.6, 1.2, 'Input x\n[B, 43]', '#f8f9fa')
_box(ax, 2.3, 1.6, 2.6, 3.0,
     'Autoencoder (DL)\nEnc 128→64→32\nDec 32→64→128→43\nweighted MSE',
     DL_FC, fw='bold')

_box(ax, 5.6, 4.5, 1.8, 0.9, 'latent z\n[B, 32]', '#f8f9fa')
_box(ax, 5.6, 3.0, 1.8, 0.9, 'recon x_hat\n[B, 43]', '#f8f9fa')
_box(ax, 5.6, 1.5, 1.8, 0.9, 'residual\n|x − x_hat|\n[B, 43]', '#fde2e4')
_box(ax, 5.6, 0.0, 1.8, 0.9, 'recon error\nMSE → gate', '#fde2e4')

_box(ax, 8.0, 1.7, 2.2, 2.6,
     'Concatenate\n[x ‖ z ‖ |x − x_hat|]\n[B, 118]', '#f1f3f5', fw='bold')
_box(ax, 10.5, 2.0, 2.0, 2.0,
     'XGBoost (ML)\n400 trees, depth 6\nmulti:softprob',
     ML_FC, fw='bold')
_box(ax, 10.5, 0.0, 2.0, 1.4,
     'Stage-1 gate\nshort-circuit\nto Normal',
     '#ffe8d6')

_box(ax, 12.7, 2.5, 0.3, 1.0, '', OUT_FC)
ax.text(13.4, 3.0, 'ŷ ∈\n{Normal,\nDoS, Probe,\nR2L, U2R}',
        ha='center', va='center', fontsize=9, fontweight='bold')

_arrow(ax, 1.8, 3.1, 2.3, 3.1)
_arrow(ax, 4.9, 4.1, 5.6, 4.95)
_arrow(ax, 4.9, 3.4, 5.6, 3.45)
_arrow(ax, 4.9, 2.6, 5.6, 1.95)
_arrow(ax, 6.5, 1.5, 6.5, 0.9)
_arrow(ax, 7.4, 4.95, 8.0, 3.6)
_arrow(ax, 7.4, 3.45, 8.0, 3.0)
_arrow(ax, 7.4, 1.95, 8.0, 2.4)
_arrow(ax, 10.2, 3.0, 10.5, 3.0)
_arrow(ax, 7.4, 0.45, 10.5, 0.7)
_arrow(ax, 12.5, 3.0, 12.7, 3.0)
_arrow(ax, 12.5, 0.7, 12.7, 2.6)

ax.text(6.5, 5.6,
        'DL provides representation (z) + per-feature error signal (residual);  '
        'ML provides supervised attack-family classification + interpretability',
        ha='center', fontsize=10, style='italic', color='#555')
ax.set_title('Model C — Cascaded AE + XGBoost', fontsize=13, fontweight='bold')

leg = [mpatches.Patch(color=DL_FC, label='Deep Learning'),
       mpatches.Patch(color=ML_FC, label='Machine Learning'),
       mpatches.Patch(color='#fde2e4', label='DL→ML feature transfer'),
       mpatches.Patch(color='#ffe8d6', label='Cascade gate'),
       mpatches.Patch(color=OUT_FC, label='Output')]
ax.legend(handles=leg, loc='lower left', fontsize=9, frameon=True)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'diagram_model_C.png'),
            dpi=160, bbox_inches='tight')
plt.show()

## 4. Cross-phase metric bars

In [ ]:
plot_metric_bars(
    full[['model', 'f1']], metric='f1',
    title='F1 across Phase 1 / Phase 2 / Phase 3 (NSL-KDD test)',
    path=os.path.join(FIGURES_DIR, 'phase_all_f1.png'),
)
plot_metric_bars(
    full[['model', 'auc']], metric='auc',
    title='ROC-AUC across Phase 1 / Phase 2 / Phase 3 (NSL-KDD test)',
    path=os.path.join(FIGURES_DIR, 'phase_all_auc.png'),
    color='#9b59b6',
)

## 5. Generate the written report

In [ ]:
deltas = pd.read_csv(os.path.join(RESULTS_DIR, 'ablation_phase3_deltas.csv'))
best_p1 = full[full['phase'] == 'Phase 1'].nlargest(1, 'f1').iloc[0]
best_p2 = full[full['phase'] == 'Phase 2'].nlargest(1, 'f1').iloc[0]
best_p3 = full[full['phase'] == 'Phase 3'].nlargest(1, 'f1').iloc[0]

shap_share = np.load(os.path.join(RESULTS_DIR, 'modelC_scores.npz'))['shap_block_share']
shap_str = (f'raw {shap_share[0]:.1f}% / latent {shap_share[1]:.1f}% / residuals {shap_share[2]:.1f}%')

report = f'''# Phase 3 Report — Hybrid ML + DL Intrusion Detection on NSL-KDD

## 1. Problem and Motivation
Phase 1 delivered statistical / unsupervised ML baselines (Z-Score, Isolation Forest)
and Phase 2 delivered deep one-class detectors (Autoencoder, β-VAE). Phase 2 surfaced
a concrete weakness: deep models reached ROC-AUC ≈ 0.93 but their reconstruction-error
threshold over-flagged benign traffic, dropping precision to ≈ 0.58. Phase 3 fixes this
by *coupling* DL feature extraction with an ML decision mechanism, in two complementary
architectures.

## 2. Hybrid Architectures
**Model A — Deep Isolation Forest.** AE encodes a record into a 32-D latent vector;
Isolation Forest is fit on those embeddings rather than on the raw 43-D input. The DL
component tames IF\'s curse-of-dimensionality on heterogeneous tabular features, and the
ML component replaces the AE\'s brittle MSE threshold with a scale-free path-length
statistic. See `figures/diagram_model_A.png`.

**Model C — Cascaded AE + XGBoost.** XGBoost is trained on a hybrid feature vector
`[ raw_features ‖ AE_latent_z ‖ |x − x_hat| ]`. The per-feature residual is the novel
transfer signal — it tells the ML classifier exactly which dimensions the DL component
could not reconstruct. The AE additionally acts as a Stage-1 gate that short-circuits
obvious normal traffic. See `figures/diagram_model_C.png`.

## 3. Methodology
- NSL-KDD train/test loaded from `Phase 1/data/`.
- Categorical encoders fit on train data only; unseen test categories → −1.
- Engineered features: `bytes_ratio`, `error_rate_diff`, `srv_diversity`.
- `StandardScaler` fit on train-normal only.
- AE trains on 80 % of train-normal (one-class regime).
- XGBoost trains on the held-out val-normal + every train attack with multi-class labels.
- Threshold tuned on a fixed `val_mixed` split by max-F1.
- Test set is touched once for final reporting.

## 4. Headline Results (NSL-KDD test)
| Phase | Best model | Accuracy | Precision | Recall | F1 | ROC-AUC |
|---|---|---|---|---|---|---|
| Phase 1 | {best_p1['model']} | {best_p1['accuracy']:.4f} | {best_p1['precision']:.4f} | {best_p1['recall']:.4f} | {best_p1['f1']:.4f} | {best_p1['auc']:.4f} |
| Phase 2 | {best_p2['model']} | {best_p2['accuracy']:.4f} | {best_p2['precision']:.4f} | {best_p2['recall']:.4f} | {best_p2['f1']:.4f} | {best_p2['auc']:.4f} |
| Phase 3 | {best_p3['model']} | {best_p3['accuracy']:.4f} | {best_p3['precision']:.4f} | {best_p3['recall']:.4f} | {best_p3['f1']:.4f} | {best_p3['auc']:.4f} |

Full table: `results/phase3_full_comparison.csv`. All ROC curves: `figures/phase3_roc_final.png`.

## 5. Ablation — what the hybrid actually contributes
Eight conditions evaluated on the same test set (`results/ablation_phase3.csv`).
Per-component F1 deltas (`results/ablation_phase3_deltas.csv`):

| Change | ΔF1 |
|---|---|
''' + ''.join(f'| {r["change"]} | {r["f1_delta"]:+.4f} |\n' for _, r in deltas.iterrows()) + f'''

Diagnostic interpretation:
- *Latent z alone* and *residuals alone* each lift XGBoost on top of the raw-only baseline,
  but they lift it in different directions — `z` improves separability of related attack
  families, while the residuals catch records the AE could not reconstruct.
- The full Model C (`raw + z + residuals`) outperforms both partial variants, confirming
  the DL features are non-redundant.
- The AE Stage-1 gate trades a small recall hit for a precision lift on the cascade, which
  is the operationally desirable direction for an IDS.
- Deep IF beats both raw IF and AE-only by replacing the brittle MSE threshold with a
  better-calibrated isolation score on the AE\'s compressed latent space.

## 6. Where does XGBoost actually look?
Block-level SHAP importance share on a 2 000-row test subsample:  
**{shap_str}**

The latent and residual blocks together account for a non-trivial fraction of the
classifier\'s decision, which is the empirical evidence that the hybrid is not a glue:
the DL-derived features carry decision-relevant signal that the raw features alone do not.
See `figures/phase3_cax_shap_blocks.png`.

## 7. Limitations and Future Work
- Threshold tuning still depends on the validation mix; deployed systems often need
  cost-sensitive thresholds rather than F1-optimal ones.
- The cascade gate currently uses a single AE threshold; a learnt gate (e.g. logistic
  regression on `[recon_error, max_z]`) would be a tidy follow-up.
- NSL-KDD is dated; cross-dataset evaluation on UNSW-NB15 / CIC-IDS-2017 would
  test generalisation.

## 8. Reproducibility
- `requirements.txt` pinned, deterministic seeds (`SEED = 42`), no hardcoded paths.
- Run order: `02 → 03 → 04 → 05 → 06`.
- Streamlit demo: `streamlit run app/streamlit_app.py`.
- Container: `docker build -t phase3-ids -f app/Dockerfile . && docker run -p 8501:8501 phase3-ids`.
'''

report_path = os.path.join(RESULTS_DIR, 'report.md')
with open(report_path, 'w') as f:
    f.write(report)
print(f'Wrote report -> {report_path}  ({len(report)} chars)')